# F1 Pit Stops Prediction - Feature Engineering

이 노트북은 **Predicting F1 Pit Stops** 경진대회의 학습 및 평가 데이터셋을 대상으로, EDA 결과와 F1 도메인 지식을 반영한 정교한 파생 피처들을 생성하는 파이프라인을 구축합니다.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

train_path = "./train.csv"
test_path = "./test.csv"

print("Loading datasets...")
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
print(f"Train shape: {train.shape}, Test shape: {test.shape}")

Loading datasets...
Train shape: (439140, 16), Test shape: (188165, 15)


## 1. 타이어 컴파운드별 소모도 피처 (Tyre Wear Features)

EDA에서 파악한 컴파운드별 실제 피트인 랩 수(`Expected_TyreLife`) 정보를 활용해 현재 타이어의 소모 비율 및 남은 수명을 유도합니다.

In [2]:
def add_tyre_wear_features(df):
    df = df.copy()
    # 컴파운드별 피트스탑 시점의 TyreLife 평균치 매핑 (EDA 결과 기반)
    expected_life = {
        'SOFT': 12.6,
        'MEDIUM': 17.1,
        'HARD': 21.4,
        'INTERMEDIATE': 18.0,
        'WET': 13.0
    }
    
    # 기대 수명 피처 생성
    df['Expected_TyreLife'] = df['Compound'].map(expected_life)
    
    # TyreLife 비율 (1.0에 가까울수록 피트스탑 한계에 도달)
    df['TyreLife_Ratio'] = df['TyreLife'] / df['Expected_TyreLife']
    
    # 남은 기대 수명
    df['TyreLife_Remaining'] = df['Expected_TyreLife'] - df['TyreLife']
    
    return df

train_fe = add_tyre_wear_features(train)
test_fe = add_tyre_wear_features(test)
print("Tyre wear features added successfully.")

Tyre wear features added successfully.


## 2. 서킷별 특성 정규화 및 통계 피처 (Circuit Features)

서킷(`Race`)마다 고유한 특성(총 랩 수, 트랙 마모도 등)이 존재하므로, 서킷 기준의 랩타임 대비율 및 타이어 수명 지표를 생성합니다.

In [3]:
# 서킷별 기준 통계량은 Train 세트에서만 계산하여 Test 세트에 매핑합니다 (Data Leakage 방지).
# 서킷별 랩타임 평균 계산
race_laptime_mean = train_fe.groupby('Race')['LapTime (s)'].mean().to_dict()

def add_circuit_features(df, race_laptime_mean):
    df = df.copy()
    
    # 서킷별 평균 랩타임 매핑
    df['Race_Avg_LapTime'] = df['Race'].map(race_laptime_mean)
    
    # 서킷 평균 대비 현재 차량의 랩타임 비율 (1.0보다 크면 상대적으로 페이스가 느려진 상태)
    df['LapTime_Ratio'] = df['LapTime (s)'] / df['Race_Avg_LapTime']
    
    # 서킷 내 상대적 순위 변화율
    # Position_Change가 양수면 추월당한 상태, 음수면 추월한 상태
    df['Relative_Position_Change'] = df['Position_Change'] / (df['Position'] + 1e-5)
    
    return df

train_fe = add_circuit_features(train_fe, race_laptime_mean)
test_fe = add_circuit_features(test_fe, race_laptime_mean)
print("Circuit features added successfully.")

Circuit features added successfully.


## 3. 페이스 트렌드 및 차량 성능 저하 피처 (Degradation & Trend Features)

차량의 노화 지표(`Cumulative_Degradation`)와 이전 몇 랩 동안의 추세를 결합합니다. 순차 연산을 위해 데이터를 `Year`, `Race`, `Driver`, `LapNumber` 기준으로 먼저 정렬합니다.

In [4]:
def add_trend_features(df):
    df = df.copy()
    # 드라이버별 시퀀스를 만들기 위해 정렬
    df = df.sort_values(by=['Year', 'Race', 'Driver', 'LapNumber']).reset_index(drop=True)
    
    # 1. 페이스 저하율: TyreLife 한 단위당 노화 누적 속도
    # TyreLife가 0인 경우(피트스탑 직후) 나눗셈 에러 방지를 위해 +1 처리
    df['Degradation_Per_Lap'] = df['Cumulative_Degradation'] / (df['TyreLife'] + 1)
    
    # 2. 직전 3랩 동안의 LapTime 평균 추세 (이동 평균)
    df['LapTime_Rolling_3'] = df.groupby(['Year', 'Race', 'Driver'])['LapTime (s)'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
    
    # 3. 이동 평균 랩타임 대비 현재 랩타임의 변화량 (페이스가 급격히 떨어지는지 포착)
    df['LapTime_Diff_Rolling'] = df['LapTime (s)'] - df['LapTime_Rolling_3']
    
    # 4. 직전 3랩 동안의 LapTime_Delta 누적치
    df['LapTime_Delta_Sum_3'] = df.groupby(['Year', 'Race', 'Driver'])['LapTime_Delta'].transform(lambda x: x.rolling(window=3, min_periods=1).sum())
    
    return df

train_fe = add_trend_features(train_fe)
test_fe = add_trend_features(test_fe)
print("Trend and degradation features added successfully.")

Trend and degradation features added successfully.


## 4. 범주형 변수 인코딩 및 처리 (Categorical Handling)

GBDT 모델 및 딥러닝 임베딩 레이어 구성을 위해 범주형 변수를 Label Encoding 처리합니다.

In [5]:
from sklearn.preprocessing import LabelEncoder

cat_cols = ['Driver', 'Compound', 'Race']

for col in cat_cols:
    le = LabelEncoder()
    # Train과 Test 전체를 커버하기 위해 합쳐서 fit을 수행합니다.
    full_series = pd.concat([train_fe[col], test_fe[col]], axis=0).astype(str)
    le.fit(full_series)
    
    train_fe[col] = le.transform(train_fe[col].astype(str))
    test_fe[col] = le.transform(test_fe[col].astype(str))
    print(f"Label encoded {col}. Classes count: {len(le.classes_)}")

Label encoded Driver. Classes count: 887
Label encoded Compound. Classes count: 5
Label encoded Race. Classes count: 26


## 5. 데이터 정합성 검증 및 내보내기

생성된 파생 피처 중 결측치(`NaN`)나 무한대(`Inf`)가 존재할 경우 제거하거나 적절한 값으로 대체합니다.

In [6]:
# 파생 피처에서 생성된 결측치 확인
print("--- Train FE Nulls ---")
print(train_fe.isnull().sum()[train_fe.isnull().sum() > 0])

print("\n--- Test FE Nulls ---")
print(test_fe.isnull().sum()[test_fe.isnull().sum() > 0])

# 무한대 값을 NaN으로 변환 후 0으로 채우기
train_fe = train_fe.replace([np.inf, -np.inf], np.nan).fillna(0)
test_fe = test_fe.replace([np.inf, -np.inf], np.nan).fillna(0)

print(f"\nFinal Train Shape: {train_fe.shape}")
print(f"Final Test Shape: {test_fe.shape}")

# 파일 저장
train_fe.to_csv("./train_fe.csv", index=False)
test_fe.to_csv("./test_fe.csv", index=False)
print("\nFeatures exported successfully to train_fe.csv and test_fe.csv!")

--- Train FE Nulls ---
Series([], dtype: int64)

--- Test FE Nulls ---
Series([], dtype: int64)

Final Train Shape: (439140, 26)
Final Test Shape: (188165, 25)

Features exported successfully to train_fe.csv and test_fe.csv!
